# AirSense — Data Understanding & Initial Data Quality Assessment

Notebook ini digunakan untuk memahami struktur dan kualitas awal data kualitas udara AirSense sebelum masuk ke tahap Exploratory Data Analysis (EDA), perhitungan ISPU, dan pemodelan.

### Tujuan
1. Memahami struktur dan tipe data.
2. Memeriksa rentang serta konsistensi timestamp.
3. Mengidentifikasi missing value dan duplicate.
4. Memeriksa nilai yang berpotensi tidak wajar.
5. Melihat statistik deskriptif awal setiap parameter.
6. Menentukan kesiapan data untuk tahap analisis berikutnya.

> **Catatan:** Pada tahap pengembangan awal digunakan data dummy untuk membangun dan menguji pipeline. Analisis final akan menggunakan data riil dari perangkat IoT.

# Import Library

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:.2f}")

print("Library berhasil di-import.")

Library berhasil di-import.


## 1. Load Dataset

Dataset yang digunakan pada tahap awal adalah data dummy `tb_konsentrasi_gas` yang memiliki struktur yang sama dengan data konsentrasi sensor AirSense.

In [2]:
DATA_PATH = Path("../../dummy_data/output/dummy_tb_konsentrasi_gas.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Dataset tidak ditemukan: {DATA_PATH.resolve()}")

df = pd.read_csv(DATA_PATH)

print("Dataset berhasil dimuat.")
print(f"Jumlah baris : {df.shape[0]:,}")
print(f"Jumlah kolom : {df.shape[1]}")

Dataset berhasil dimuat.
Jumlah baris : 10,081
Jumlah kolom : 8


## 2. Struktur Dataset

Pemeriksaan awal dilakukan untuk melihat contoh data, nama kolom, dimensi dataset, serta tipe data masing-masing variabel.

In [3]:
df.head()

,created_at,pm25_ugm3,pm10_ugm3,co_ugm3,no2_ugm3,o3_ugm3,temperature,humidity
0,2026-08-30T13:49:00+00:00,11.31,14.87,2682.45,0.42,20.61,38.99,48.40
1,2026-08-30T13:50:00+00:00,16.18,21.71,2673.63,0.00,73.92,36.95,52.98
2,2026-08-30T13:51:00+00:00,15.33,17.59,2818.74,0.00,21.04,37.30,52.37
3,2026-08-30T13:52:00+00:00,13.20,9.97,2548.78,0.43,21.05,38.50,49.88
4,2026-08-30T13:53:00+00:00,16.70,17.53,2537.62,0.00,80.64,38.24,53.81


In [4]:
print("Dimensi dataset:", df.shape)

print("\nNama kolom:")
for i, col in enumerate(df.columns, start=1):
    print(f"{i}. {col}")

print("\nTipe data:")
print(df.dtypes)

Dimensi dataset: (10081, 8)

Nama kolom:
1. created_at
2. pm25_ugm3
3. pm10_ugm3
4. co_ugm3
5. no2_ugm3
6. o3_ugm3
7. temperature
8. humidity

Tipe data:
created_at      object
pm25_ugm3      float64
pm10_ugm3      float64
co_ugm3        float64
no2_ugm3       float64
o3_ugm3        float64
temperature    float64
humidity       float64
dtype: object


In [ ]:
#Cek Informasi Dataset
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10081 entries, 0 to 10080
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   created_at   10081 non-null  object 
 1   pm25_ugm3    10081 non-null  float64
 2   pm10_ugm3    10081 non-null  float64
 3   co_ugm3      10081 non-null  float64
 4   no2_ugm3     10081 non-null  float64
 5   o3_ugm3      10081 non-null  float64
 6   temperature  10081 non-null  float64
 7   humidity     10081 non-null  float64
dtypes: float64(7), object(1)
memory usage: 630.2+ KB


Tujuannya melihat apakah parameter polutan sudah terbaca sebagai numerik dan apakah created_at masih object.

Kalau created_at masih object, itu normal saat baru membaca CSV. Nanti kita konversi ke datetime.

## 3. Validasi Timestamp

Kolom `created_at` diperiksa untuk memastikan seluruh timestamp dapat dikonversi ke format datetime, tersusun secara kronologis, serta memiliki interval pencatatan yang konsisten.

In [6]:
# Konversi created_at menjadi datetime
df["created_at"] = pd.to_datetime(
    df["created_at"],
    errors="coerce",
    utc=True
)

# Urutkan berdasarkan waktu
df = df.sort_values("created_at").reset_index(drop=True)

print("Tipe data created_at :", df["created_at"].dtype)
print("Invalid timestamp    :", df["created_at"].isna().sum())

print("\nRentang waktu:")
print("Awal  :", df["created_at"].min())
print("Akhir :", df["created_at"].max())

duration = df["created_at"].max() - df["created_at"].min()
print("Durasi:", duration)

Tipe data created_at : datetime64[ns, UTC]
Invalid timestamp    : 0

Rentang waktu:
Awal  : 2026-08-30 13:49:00+00:00
Akhir : 2026-09-06 13:49:00+00:00
Durasi: 7 days 00:00:00


In [7]:
#Cek Interval Pencatatan
time_diff = df["created_at"].diff()

print("Distribusi interval antar-record:")
print(time_diff.value_counts().head(10))

Distribusi interval antar-record:
created_at
0 days 00:01:00    10080
Name: count, dtype: int64


In [8]:
expected_interval = pd.Timedelta(minutes=1)

irregular_interval = (
    time_diff.notna() &
    (time_diff != expected_interval)
)

print("Jumlah interval tidak 1 menit :", irregular_interval.sum())
print("Persentase interval tidak sesuai:",
      f"{irregular_interval.mean() * 100:.2f}%")

Jumlah interval tidak 1 menit : 0
Persentase interval tidak sesuai: 0.00%


## 4. Pemeriksaan Missing Value

Pemeriksaan dilakukan untuk mengetahui apakah terdapat nilai yang tidak tersedia pada masing-masing variabel. Pada tahap ini data yang hilang hanya diidentifikasi dan belum dilakukan penghapusan atau imputasi.

In [9]:
missing_count = df.isna().sum()
missing_percent = (missing_count / len(df) * 100).round(2)

missing_summary = pd.DataFrame({
    "Missing Count": missing_count,
    "Missing (%)": missing_percent
})

missing_summary

,Missing Count,Missing (%)
created_at,0,0.00
pm25_ugm3,0,0.00
pm10_ugm3,0,0.00
co_ugm3,0,0.00
no2_ugm3,0,0.00
o3_ugm3,0,0.00
temperature,0,0.00
humidity,0,0.00


## 5. Pemeriksaan Duplicate

Pemeriksaan duplicate dilakukan terhadap keseluruhan baris dan timestamp untuk mengidentifikasi kemungkinan pencatatan data yang berulang.

In [10]:
duplicate_rows = df.duplicated().sum()
duplicate_timestamp = df["created_at"].duplicated().sum()

print("Duplicate seluruh baris :", duplicate_rows)
print("Duplicate timestamp     :", duplicate_timestamp)

Duplicate seluruh baris : 0
Duplicate timestamp     : 0


## 6. Statistik Deskriptif Awal

Statistik deskriptif digunakan untuk memperoleh gambaran awal mengenai distribusi nilai setiap parameter, meliputi jumlah observasi, rata-rata, standar deviasi, nilai minimum, kuartil, median, dan maksimum.

In [11]:
numeric_cols = [
    "pm25_ugm3",
    "pm10_ugm3",
    "co_ugm3",
    "no2_ugm3",
    "o3_ugm3",
    "temperature",
    "humidity"
]

df[numeric_cols].describe().T

,count,mean,std,min,25%,50%,75%,max
pm25_ugm3,10081.00,15.89,6.44,8.00,13.26,15.27,17.31,84.23
pm10_ugm3,10081.00,18.20,7.48,7.50,14.57,17.50,20.47,99.77
co_ugm3,10081.00,2950.41,417.22,2192.88,2691.62,2873.32,3160.60,7308.76
no2_ugm3,10081.00,0.78,11.26,0.00,0.00,0.00,0.06,242.03
o3_ugm3,10081.00,26.37,14.79,19.63,19.67,20.35,21.01,88.85
temperature,10081.00,34.96,2.01,29.83,33.33,34.69,36.62,40.89
humidity,10081.00,60.95,6.04,39.77,56.54,61.26,65.39,78.40


## 7. Pemeriksaan Nilai Berpotensi Tidak Wajar

Pemeriksaan awal dilakukan untuk mengidentifikasi nilai yang memerlukan investigasi lebih lanjut. Nilai yang terdeteksi tidak langsung dihapus karena dapat berasal dari error sensor maupun kondisi lingkungan yang sebenarnya.

In [12]:
negative_check = (df[numeric_cols] < 0).sum()

negative_summary = pd.DataFrame({
    "Jumlah Nilai Negatif": negative_check
})

negative_summary

,Jumlah Nilai Negatif
pm25_ugm3,0
pm10_ugm3,0
co_ugm3,0
no2_ugm3,0
o3_ugm3,0
temperature,0
humidity,0


## 8. Pemeriksaan Nilai Nol

Pemeriksaan nilai nol dilakukan untuk mengetahui parameter yang memiliki proporsi nilai nol tinggi. Nilai nol tidak langsung dianggap sebagai missing value atau error karena dapat berkaitan dengan karakteristik pembacaan sensor.

In [13]:
zero_count = (df[numeric_cols] == 0).sum()
zero_percent = (zero_count / len(df) * 100).round(2)

zero_summary = pd.DataFrame({
    "Jumlah Nilai 0": zero_count,
    "Persentase (%)": zero_percent
})

zero_summary

,Jumlah Nilai 0,Persentase (%)
pm25_ugm3,0,0.00
pm10_ugm3,0,0.00
co_ugm3,0,0.00
no2_ugm3,7434,73.74
o3_ugm3,0,0.00
temperature,0,0.00
humidity,0,0.00


## 9. Pemeriksaan Nilai Dominan

Pemeriksaan dilakukan untuk mengetahui apakah suatu parameter memiliki nilai tertentu yang muncul dengan frekuensi sangat tinggi. Kondisi ini dapat menjadi karakteristik sensor, pembulatan, batas pembacaan, atau karakteristik data itu sendiri.

In [14]:
dominant_summary = []

for col in numeric_cols:
    counts = df[col].value_counts()
    
    dominant_summary.append({
        "Parameter": col,
        "Nilai Paling Sering": counts.index[0],
        "Frekuensi": counts.iloc[0],
        "Persentase (%)": round(counts.iloc[0] / len(df) * 100, 2),
        "Jumlah Nilai Unik": df[col].nunique()
    })

dominant_summary = pd.DataFrame(dominant_summary)

dominant_summary

,Parameter,Nilai Paling Sering,Frekuensi,Persentase (%),Jumlah Nilai Unik
0,pm25_ugm3,8.00,185,1.84,1631
1,pm10_ugm3,7.50,79,0.78,2093
2,co_ugm3,2738.61,4,0.04,9534
3,no2_ugm3,0.00,7434,73.74,203
4,o3_ugm3,19.63,2384,23.65,1914
5,temperature,33.14,33,0.33,908
6,humidity,58.92,15,0.15,2520


## 10. Screening Nilai Ekstrem

Metode Interquartile Range (IQR) digunakan sebagai screening awal untuk mengidentifikasi observasi yang berada jauh dari distribusi utama data. Hasil screening tidak langsung dianggap sebagai error dan tidak dihapus secara otomatis, karena nilai ekstrem dapat merepresentasikan kejadian lingkungan yang sebenarnya.

In [15]:
extreme_summary = []

for col in numeric_cols:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1

    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    extreme_mask = (
        (df[col] < lower_bound) |
        (df[col] > upper_bound)
    )

    extreme_summary.append({
        "Parameter": col,
        "Lower Bound": round(lower_bound, 2),
        "Upper Bound": round(upper_bound, 2),
        "Jumlah Kandidat": extreme_mask.sum(),
        "Persentase (%)": round(extreme_mask.mean() * 100, 2)
    })

extreme_summary = pd.DataFrame(extreme_summary)

extreme_summary

,Parameter,Lower Bound,Upper Bound,Jumlah Kandidat,Persentase (%)
0,pm25_ugm3,7.19,23.38,252,2.50
1,pm10_ugm3,5.72,29.32,216,2.14
2,co_ugm3,1988.15,3864.07,76,0.75
3,no2_ugm3,-0.09,0.15,2283,22.65
4,o3_ugm3,17.66,23.02,1974,19.58
5,temperature,28.40,41.55,0,0.00
6,humidity,43.26,78.67,4,0.04


In [16]:
#Buat flag tanpa mengubah data
df_quality = df.copy()

for col in numeric_cols:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1

    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    df_quality[f"{col}_extreme_flag"] = (
        (df[col] < lower) |
        (df[col] > upper)
    )

df_quality.head()

,created_at,pm25_ugm3,pm10_ugm3,co_ugm3,no2_ugm3,o3_ugm3,temperature,humidity,pm25_ugm3_extreme_flag,pm10_ugm3_extreme_flag,co_ugm3_extreme_flag,no2_ugm3_extreme_flag,o3_ugm3_extreme_flag,temperature_extreme_flag,humidity_extreme_flag
0,2026-08-30 13:49:00+00:00,11.31,14.87,2682.45,0.42,20.61,38.99,48.40,False,False,False,True,False,False,False
1,2026-08-30 13:50:00+00:00,16.18,21.71,2673.63,0.00,73.92,36.95,52.98,False,False,False,False,True,False,False
2,2026-08-30 13:51:00+00:00,15.33,17.59,2818.74,0.00,21.04,37.30,52.37,False,False,False,False,False,False,False
3,2026-08-30 13:52:00+00:00,13.20,9.97,2548.78,0.43,21.05,38.50,49.88,False,False,False,True,False,False,False
4,2026-08-30 13:53:00+00:00,16.70,17.53,2537.62,0.00,80.64,38.24,53.81,False,False,False,False,True,False,False


## 11. Kesimpulan Data Understanding

Berdasarkan pemeriksaan awal terhadap dataset dummy AirSense, diperoleh beberapa temuan:

- Dataset terdiri dari 10.081 observasi dengan periode pengamatan selama 7 hari dan interval pencatatan konsisten setiap 1 menit.
- Tidak ditemukan invalid timestamp, missing value, duplicate record, duplicate timestamp, maupun nilai negatif.
- Seluruh parameter sensor telah tersimpan dalam tipe data numerik, sedangkan timestamp telah berhasil dikonversi menjadi format datetime.
- NO₂ memiliki karakteristik distribusi khusus, dengan 73,74% observasi bernilai 0.
- O₃ memiliki konsentrasi nilai yang tinggi pada 19,63 µg/m³, sehingga pola ini perlu diperhatikan pada analisis berikutnya.
- Screening menggunakan IQR menemukan sejumlah nilai ekstrem pada beberapa parameter. Namun, nilai tersebut tidak langsung dianggap sebagai kesalahan data karena dapat merepresentasikan karakteristik sensor maupun kejadian tertentu.
- Oleh karena itu, pendekatan yang digunakan adalah mendeteksi dan menandai kandidat nilai ekstrem tanpa melakukan penghapusan otomatis.
- Karena dataset yang digunakan merupakan data dummy, hasil pemeriksaan ini digunakan untuk validasi pipeline pengolahan data dan belum digunakan untuk menarik kesimpulan mengenai kondisi kualitas udara sebenarnya.

Secara struktural, dataset dapat dilanjutkan ke tahap Exploratory Data Analysis (EDA).